In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from src.data_loader import load_data
from src.data_quality import summary_quality
from src.metrics import calculate_wape, calculate_bias, calculate_metrics, plot_wape_cd_horizon, calculate_out_of_plan_metrics, calculate_client_plan_bias, plot_client_plan_bias, calculate_client_plan_adherence
from src.forecast import prepare_forecast_features, select_backtest_dates, create_forecast_pipeline,predict_rolling_origin, evaluate_minimum_history, train_model_with_gate, generate_final_forecast

# Lectura de Data

In [ ]:
path_pedidos_historicos_genericos = 'data/pedidos_historicos_genericos.csv'
path_pronostico_demanda_generico = 'data/pronostico_demanda_generico.csv'
path_capacidad_logistica_generica = 'data/capacidad_logistica_generica.csv'
path_plan_entrega_clientes_generico = 'data/plan_entrega_clientes_generico.csv'


In [ ]:
pedidos_historicos_genericos = load_data(path_pedidos_historicos_genericos)

pronostico_demanda_generico = load_data(path_pronostico_demanda_generico)

capacidad_logistica_generica = load_data(path_capacidad_logistica_generica)

plan_entrega_clientes_generico = load_data(path_plan_entrega_clientes_generico)


In [ ]:
summary_df = {
    'pedidos_historicos_genericos': pedidos_historicos_genericos,
    'pronostico_demanda_generico': pronostico_demanda_generico,
    'capacidad_logistica_generica': capacidad_logistica_generica,
    'plan_entrega_clientes_generico': plan_entrega_clientes_generico
}

In [ ]:
for name, df in summary_df.items():
    print(f"*********** {name} ***********")

    quality = summary_quality(df)

    for section, result in quality.items():
        print(f"{section.capitalize()}:")
        display(result)

# Tarea 1: Diagnóstico de Calidad y Backtesting

In [ ]:
pedidos_historicos_cd = pedidos_historicos_genericos.groupby(
    ['centro_distribucion', 'fecha_despacho']).agg({'unidades_reales': 'sum'}).reset_index()

df_forecast = pronostico_demanda_generico.merge(
    pedidos_historicos_cd,
    on=["centro_distribucion", "fecha_despacho"],
    how="inner",
)

In [ ]:
metrics_cd_horizon = calculate_metrics(
    df_forecast,
    real_col="unidades_reales",
    forecast_col="unidades_pronosticadas",
    group_by=["centro_distribucion", "horizonte"]
)

metrics_cd = calculate_metrics(
    df_forecast,
    real_col="unidades_reales",
    forecast_col="unidades_pronosticadas",
    group_by=["centro_distribucion"]
)
metrics_horizon = calculate_metrics(
    df_forecast,
    real_col="unidades_reales",
    forecast_col="unidades_pronosticadas",
    group_by=["horizonte"]
)
display(metrics_cd_horizon)
display(metrics_cd)
display(metrics_horizon)

In [ ]:
plot_wape_cd_horizon(metrics_cd_horizon, metrics_horizon, horizons= ["D+1", "D+2", "D+3", "D+4"])


1. Los centros CD_SUR, CD_ESTE y CD_NORTE presentan una subestimación consistente de la demanda: su Bias es positivo tanto en el resultado agregado por centro como al desglosarlo en los cuatro horizontes de planificación, de D+1 a D+4.
Por su parte, CD_CENTRO, CD_OESTE y CD_PACIFICO presentan una sobreestimación consistente, con Bias negativo tanto en el agregado por centro como en cada horizonte.

2. A nivel global, la precisión del pronóstico disminuye a medida que aumenta el horizonte: el WAPE pasa de 19,77 % en D+1 a 20,30 % en D+2, 20,85 % en D+3 y 22,53 % en D+4. Esto representa un aumento del error de 2,76 puntos porcentuales entre D+1 y D+4.
Al desglosar por centro, el deterioro no es uniforme. CD_CENTRO y CD_PACIFICO presentan los mayores aumentos entre D+1 y D+4, de 8,08 y 5,54 puntos porcentuales, respectivamente. CD_ESTE y CD_NORTE también terminan con un error mayor en D+4, aunque la evolución entre horizontes no es estrictamente creciente.
En cambio, CD_OESTE y CD_SUR presentan un WAPE menor en D+4 que en D+1. Por ello, la degradación observada a nivel global no debe generalizarse a todos los centros.

# Tarea 2 — Estado del plan de entrega

## 1. ¿Existe  un  Bias  en  el  plan  de  entrega?  Es  decir,  ¿el  plan  (unidades  y  fechas) subestima o sobreestima sistemáticamente el comportamiento real de compra de los clientes?

In [ ]:
df = plan_entrega_clientes_generico.copy()

for columna in ["fecha_plan", "fecha_compra_real"]:
    df[columna] = pd.to_datetime(df[columna], errors="raise")

bias_clientes = calculate_client_plan_bias(df)

In [ ]:
plot_client_plan_bias(bias_clientes)

En unidades, se ve que la mayoría de los clientes tiene un Bias positivo, por lo que el plan tiende a quedarse corto respecto de lo que realmente compran. Esto no pasa con todos: también hay clientes con Bias negativo, donde se planifican más unidades de las que terminan comprando.
En fechas, muchos clientes están cerca de cero, aunque el histograma se extiende más hacia valores positivos. Esto indica que algunos clientes compran, en promedio, después de la fecha planificada. También hay quienes compran antes, así que no se observa el mismo comportamiento en todos.

## 2. ¿Cuál es el porcentaje de compras realizadas fuera del plan? Analízalo en total, por centro de distribución y por día de la semana planificado.

In [ ]:
dias = [
    "Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"
]
df["dia_semana_planificado"] = (
    df["fecha_plan"].dt.dayofweek
)
resumen_total = calculate_out_of_plan_metrics(df)

resumen_cd = calculate_out_of_plan_metrics(
    df,
    group_by=["centro_distribucion"],
)

resumen_dia = calculate_out_of_plan_metrics(
    df,
    group_by=["dia_semana_planificado"],
)

resumen_dia["dia_semana_planificado"] = (
    resumen_dia["dia_semana_planificado"]
    .map(dict(enumerate(dias)))
)

display(resumen_total)
display(resumen_cd)
display(resumen_dia)

El 17,46 % de las compras se realiza fuera de la fecha planificada. Por centro, CD_NORTE tiene el mayor porcentaje, con un 24,39 %, mientras que CD_OESTE presenta el menor, con un 10,16 %.Al revisar el día planificado, los mayores porcentajes aparecen el miércoles, con un 22,94 %, y el lunes, con un 22,12 %. El martes tiene la mayor cantidad de compras fuera del plan, pero también tiene más compras planificadas, por lo que su porcentaje es menor: 16,22 %.

## 3. ¿Qué  métricas  adicionales  calcularías  para  completar  el  panorama  operativo?  (ej. adhesión al plan por cliente, variación de fecha, variación de volumen, concentración de desvíos). Justifica por qué aportan. 



In [ ]:
resumen_clientes_plan  = calculate_client_plan_adherence(df)
resumen_clientes_plan.head(10)

In [ ]:
desvio_dias = (
    df["fecha_compra_real"] - df["fecha_plan"]
).dt.days

bias_global_plan = calculate_bias(
    df["unidades_reales"],
    df["unidades_plan"],
)

adhesion_global = desvio_dias.eq(0).mean()
desvio_medio = desvio_dias.mean()

clientes_bias_positivo = (
    bias_clientes["bias_volumen"] > 0
).sum()
total_clientes = df["cliente_id"].nunique()

total_desvios = df["fuera_de_plan"].sum()
desvios_top10 = (
    resumen_clientes_plan
    .nlargest(10, "compras_fuera_plan")["compras_fuera_plan"]
    .sum()
)
concentracion_top10 = desvios_top10 / total_desvios

participacion_martes = (
    df["fecha_plan"].dt.dayofweek.eq(1).mean()
)

resumen_indicadores_plan = pd.DataFrame({
    "Indicador": [
        "Bias global de volumen",
        "Adhesión a la fecha planificada",
        "Desvío medio de fecha",
        "Clientes con Bias de volumen positivo",
        "Desvíos concentrados en los 10 principales clientes",
        "Compras planificadas para el martes",
        "Compras posteriores a la fecha planificada",
        "Compras anteriores a la fecha planificada",
    ],
    "Resultado": [
        f"{bias_global_plan:.2%}".replace(".", ","),
        f"{adhesion_global:.2%}".replace(".", ","),
        f"{desvio_medio:.2f} días".replace(".", ","),
        f"{clientes_bias_positivo} de {total_clientes}",
        f"{concentracion_top10:.2%}".replace(".", ","),
        f"{participacion_martes:.2%}".replace(".", ","),
        str(desvio_dias.gt(0).sum()),
        str(desvio_dias.lt(0).sum()),
    ],
})

display(resumen_indicadores_plan)

Calcularía las siguientes metricas:

Adhesión al plan por cliente: calcularía el porcentaje de compras que cada cliente realiza en la fecha acordada. Esto ayudaría a identificar con qué clientes conviene revisar el plan de entrega. 

Concentración de desvíos: calcularía qué porcentaje del total de compras fuera del plan corresponde a cada cliente. Así podría ver si los desvíos se concentran en pocos clientes o están repartidos, y decidir dónde enfocar primero la revisión.

ejemplo: CLIENTE_012 presenta 6 compras fuera del plan de un total de 9, con una adhesión del 33,33 %. Además, representa el 4,65 % de todos los desvíos. Esto permite identificarlo como uno de los clientes cuyo plan convendría revisar.

## 4. Sabiendo  que  los  clientes  tienen  preferencia  por  comprar  en días  específicos de  la semana (con un pico los martes), ¿qué estrategia propondrías para aplanar la curva de demanda? Describe el mecanismo, cómo lo implementarías y qué KPI usarías para medir su efecto.

Propondría que la plataforma priorice fechas con capacidad disponible al recomendar el despacho. Si el martes tiene una carga alta, podría mostrar otras opciones dentro de los plazos de entrega permitidos, manteniendo la elección del cliente. Desde el área de datos, identificaría los días de mayor carga por centro y analizaría el historial de compra para apoyar esas recomendaciones. Las reglas de capacidad y los plazos aceptables se definirían junto con operaciones y comercial. Haría una prueba acotada y mediría qué porcentaje de clientes acepta las fechas alternativas y si disminuye la concentración del volumen los martes. También revisaría que no se genere otro pico ni empeore el cumplimiento de las entregas.

# Tarea 3 — Modelo complementario de forecast

In [ ]:
df_modelo, columnas_dummies_codificadas = prepare_forecast_features(df_hist=pedidos_historicos_cd, df_forecast_base=df_forecast)

In [ ]:
columnas_numericas = [
    "unidades_pronosticadas",
    "promedio_error_7_dias",
    "promedio_error_14_dias",
    "promedio_7_dias",
    "promedio_14_dias",
    "std_7_dias",
    "std_14_dias",
]

features = columnas_numericas + columnas_dummies_codificadas
objetivo = "diff_forecast_base_real"



In [ ]:
train, val, test, fechas_val, fechas_test = select_backtest_dates(df=df_modelo)

In [ ]:
parametros = {
    "modelo__alpha": [0.01, 0.05, 0.1, 0.5, 1.0],
    "modelo__l1_ratio": [0.1, 0.2, 0.5, 0.8, 0.9]
}

In [ ]:
pipeline, configuraciones = create_forecast_pipeline(parametros, columnas_numericas, columnas_dummies_codificadas)

In [ ]:
resultados = []
predicciones_por_configuracion = {}

for id_config, configuracion in enumerate(configuraciones):

    predicciones = predict_rolling_origin(
        df_modelo=df_modelo,
        fechas=fechas_val,
        pipeline=pipeline,
        configuracion=configuracion,
        features=features,
        objetivo=objetivo,
    )

    predicciones_por_configuracion[id_config] = predicciones

    wape_base = calculate_wape(
        predicciones["unidades_reales"],
        predicciones["unidades_pronosticadas"],
    )

    wape_corregido = calculate_wape(
        predicciones["unidades_reales"],
        predicciones["pronostico_corregido"],
    )

    resultados.append({
        "id_config": id_config,
        "alpha": configuracion["modelo__alpha"],
        "l1_ratio": configuracion["modelo__l1_ratio"],
        "wape_base": wape_base,
        "wape_corregido": wape_corregido,
        "mejora_pp": 100 * (wape_base - wape_corregido),
    })

resumen_val = (
    pd.DataFrame(resultados)
    .sort_values("wape_corregido")
    .reset_index(drop=True)
)

display(
    resumen_val.style.format({
        "wape_base": "{:.2%}",
        "wape_corregido": "{:.2%}",
        "mejora_pp": "{:.2f}",
    })
)

In [ ]:
mejor_id = int(resumen_val.loc[0, "id_config"])
mejores_parametros = configuraciones[mejor_id]

predicciones_test = predict_rolling_origin(
    df_modelo=df_modelo,
    fechas=fechas_test,
    pipeline=pipeline,
    configuracion=mejores_parametros,
    features=features,
    objetivo=objetivo,
)

wape_base_test = calculate_wape(
    predicciones_test["unidades_reales"],
    predicciones_test["unidades_pronosticadas"],
)

wape_corregido_test = calculate_wape(
    predicciones_test["unidades_reales"],
    predicciones_test["pronostico_corregido"],
)

resumen_test = pd.DataFrame([{
    "alpha": mejores_parametros["modelo__alpha"],
    "l1_ratio": mejores_parametros["modelo__l1_ratio"],
    "wape_base": wape_base_test,
    "wape_corregido": wape_corregido_test,
    "mejora_pp": 100 * (wape_base_test - wape_corregido_test),
}])

display(
    resumen_test.style.format({
        "wape_base": "{:.2%}",
        "wape_corregido": "{:.2%}",
        "mejora_pp": "{:.2f}",
    })
)

Se utilizó un único modelo Elastic Net, incorporando el centro de distribución y el horizonte como variables mediante one-hot encoding. Esto evitó entrenar 24 modelos separados, uno por cada combinación de los 6 centros y los 4 horizontes. Así se aprovechan conjuntamente las observaciones disponibles, sin dividir un historial limitado entre muchos modelos. La codificación permite considerar diferencias entre centros y horizontes, mientras el modelo aprende relaciones comunes con toda la información de entrenamiento.El modelo estima la diferencia entre el pronóstico base y las unidades reales. Para ello utiliza el pronóstico base, el día de la semana, el centro, el horizonte y estadísticas móviles de demanda y error de 7 y 14 días. La diferencia predicha se resta del pronóstico base para obtener el pronóstico corregido.Para evaluar el modelo se reservaron cuatro fechas de emisión para validación y las últimas tres fechas elegibles para test. La evaluación se hizo avanzando por fecha: en cada una se volvió a entrenar el modelo y a ajustar el escalado usando únicamente resultados que ya estaban disponibles. Se asumió que las unidades reales de un despacho se conocen al día siguiente.En validación se probaron distintas combinaciones de parámetros y se seleccionó la de menor WAPE: alpha = 0,05 y l1_ratio = 0,2. Esa configuración se mantuvo para las fechas de test, que no se utilizaron para elegir los parámetros.El modelo mejoró la precisión en el período evaluado. En validación, el WAPE disminuyó de 19,62 % a 7,69 %, una mejora de 11,93 puntos porcentuales. En test, bajó de 20,38 % a 7,76 %, equivalente a 12,62 puntos porcentuales. En ambas evaluaciones se compararon el pronóstico base y el corregido sobre los mismos registros. 

# Tarea 4: Regla de Resiliencia

In [ ]:
estado_centros = evaluate_minimum_history(
    df_hist=pedidos_historicos_genericos,
)


In [ ]:
modelo_final = train_model_with_gate(
    df_modelo=df_modelo,
    estado_centros=estado_centros,
    pipeline=pipeline,
    configuracion=mejores_parametros,
    features=features,
    objetivo=objetivo,
)

In [ ]:
claves = [
    "centro_distribucion",
    "fecha_emision",
    "fecha_despacho",
    "horizonte",
]

df_prediccion = pronostico_demanda_generico.copy()
df_variables = df_modelo.copy()

for columna in ["fecha_emision", "fecha_despacho"]:
    df_prediccion[columna] = pd.to_datetime(
        df_prediccion[columna]
    ).dt.normalize()

    df_variables[columna] = pd.to_datetime(
        df_variables[columna]
    ).dt.normalize()

columnas_agregar = [
    columna
    for columna in features
    if columna not in df_prediccion.columns
]

df_prediccion = df_prediccion.merge(
    df_variables[claves + columnas_agregar],
    on=claves,
    how="left",
    validate="one_to_one",
)

In [ ]:
predicciones_finales = generate_final_forecast(
    df_prediccion=df_prediccion,
    estado_centros=estado_centros,
    modelo=modelo_final,
    features=features,
)

predicciones_finales[[
    "centro_distribucion",
    "horizonte",
    "fecha_despacho",
    "unidades_pronosticadas",
    "pronostico_final",
    "fuente_pronostico",
]]


Se implementó un control de historia mínima por centro de distribución que habilita el entrenamiento del modelo complementario cuando existen al menos 45 días consecutivos de datos. Los centros que no cumplen esta condición mantienen el pronóstico base como respaldo. Al cierre del archivo, los seis centros cuentan con 60 días continuos y quedan habilitados. El umbral de 45 días permite disponer de seis semanas completas de observaciones para representar el comportamiento semanal de la demanda. Reducirlo a 30 días adelantaría la habilitación en 15 días, pero limitaría la historia mínima a cuatro semanas completas, aumentando el riesgo de aprender patrones poco estables. Elevarlo a 60 días aportaría ocho semanas completas, aunque retrasaría la habilitación en 15 días y prolongaría el uso del pronóstico base y sus posibles sesgos.

# Tarea 5: Arquitectura e Infraestructura Cloud (AWS)

In [ ]:
from PIL import Image

img = Image.open('outputs/arquitectura_tarea_5.png')


img